<p style="background-color: #4CAF50; font-size: 48px; font-weight: bold; color: white; padding: 20px; text-align: center;">
🧹 Data Preparation 
</p>


In [15]:
#Import librairies
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math 
from scipy.stats import pearsonr
import scipy.stats as stats
from plotly.subplots import make_subplots
from scipy.stats import shapiro
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn import preprocessing
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore") 
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn import preprocessing
import plotly.graph_objects as go
import plotly.express as px
import missingno as msno
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import skew, shapiro
from scipy.stats import boxcox
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler
import itertools
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
import shap
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours
from collections import Counter
from sklearn.feature_selection import SelectKBest
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from prettytable import PrettyTable
import featuretools as ft

In [16]:
#import data
data = pd.read_csv(
    'C:/Users/Mega-Pc/Freelance/data/ecommerce/data.csv',encoding='ISO-8859-1')

## 🔄 Check for duplicates

In [3]:
duplicate_count = data.duplicated().sum()
print(f'Number of duplicate records: {duplicate_count}')


Number of duplicate records: 5268


In [4]:
# drop duplicates
data = data.drop_duplicates()

# Verify removal
duplicate_count_after = data.duplicated().sum()
print(f'Number of duplicate records after dropping: {duplicate_count_after}')

Number of duplicate records after dropping: 0


## Missing columns

In [5]:
missing_counts = data.isnull().sum()
missing_percent = (missing_counts / len(data)) * 100

missing_df = pd.DataFrame({
    'MissingCount': missing_counts,
    'MissingPercent': missing_percent
})

print(missing_df)


             MissingCount  MissingPercent
InvoiceNo               0        0.000000
StockCode               0        0.000000
Description          1454        0.270945
Quantity                0        0.000000
InvoiceDate             0        0.000000
UnitPrice               0        0.000000
CustomerID         135037       25.163377
Country                 0        0.000000


In [6]:
# Drop rows with missing CustomerID
data = data.dropna(subset=['CustomerID'])

#Fill Description missing values with unknown 

data['Description'] = data['Description'].fillna('Unknown')



In [7]:
missing_counts = data.isnull().sum()
missing_percent = (missing_counts / len(data)) * 100

missing_df = pd.DataFrame({
    'MissingCount': missing_counts,
    'MissingPercent': missing_percent
})

print(missing_df)


             MissingCount  MissingPercent
InvoiceNo               0             0.0
StockCode               0             0.0
Description             0             0.0
Quantity                0             0.0
InvoiceDate             0             0.0
UnitPrice               0             0.0
CustomerID              0             0.0
Country                 0             0.0


> **Missing Values Handling**
>
> - `Description`: missing values filled with `"Unknown"` (0.27% of rows)
> - `CustomerID`: missing values dropped (25% of rows)


## ✨ Data Transformation

## Handle outliers by capping them 

In [8]:
#data
num_cols = data.select_dtypes(include=['int64', 'float64']).columns
num_cols = [col for col in num_cols if col != 'CustomerID']
outliers_info = []

for col in num_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
    
    outliers_info.append({
        'Feature': col,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'Outliers Count': len(outliers),
        'Total Points': len(data)
    })

outliers_df = pd.DataFrame(outliers_info)
print(outliers_df)


     Feature  Lower Bound  Upper Bound  Outliers Count  Total Points
0   Quantity        -13.0         27.0           26646        401604
1  UnitPrice         -2.5          7.5           35802        401604


In [9]:
def handle_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[column] = np.clip(df[column], lower_bound, upper_bound)
    return df

for col in num_cols:
    data = handle_outliers(data, col)


In [10]:
# results after capping outliers
outliers_info = []

for col in num_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
    
    outliers_info.append({
        'Feature': col,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'Outliers Count': len(outliers),
        'Total Points': len(data)
    })

outliers_df = pd.DataFrame(outliers_info)
print(outliers_df)


     Feature  Lower Bound  Upper Bound  Outliers Count  Total Points
0   Quantity        -13.0         27.0               0        401604
1  UnitPrice         -2.5          7.5               0        401604


## Feature Engineering

In [11]:
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], errors='coerce')

# total price per row
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# description text features
data['DescriptionLength'] = data['Description'].fillna("").apply(len)
data['NumWordsInDescription'] = data['Description'].fillna("").apply(lambda x: len(x.split()))

# invoice type flag
data['IsReturn'] = (data['Quantity'] < 0).astype(int)

# temporal features
data['InvoiceHour'] = data['InvoiceDate'].dt.hour
data['InvoiceDayOfWeek'] = data['InvoiceDate'].dt.dayofweek
data['InvoiceMonth'] = data['InvoiceDate'].dt.month
data['InvoiceQuarter'] = data['InvoiceDate'].dt.quarter
data['IsWeekend'] = data['InvoiceDayOfWeek'].isin([5,6]).astype(int)
data['IsHolidaySeason'] = data['InvoiceMonth'].isin([11,12]).astype(int)

# invoice-level aggregation 
invoice_agg = data.groupby('InvoiceNo').agg(
    NumUniqueItems=('StockCode','nunique'),
    TotalQuantity=('Quantity','sum'),
    AvgPricePerItem=('TotalPrice', lambda x: x.sum()/x.count())
).reset_index()

data = data.merge(invoice_agg, on='InvoiceNo', how='left')

# customer-level aggregation 
customer_agg = data.groupby('CustomerID').agg(
    NumTransactions=('InvoiceNo','nunique'),
    TotalSpent=('TotalPrice','sum'),
    AvgTransactionValue=('TotalPrice','mean'),
    MaxTransactionValue=('TotalPrice','max'),
    AvgItemsPerTransaction=('Quantity','mean'),
    ReturnRate=('IsReturn','mean'),
    NumUniqueItemsBought=('StockCode','nunique'),
    FirstPurchase=('InvoiceDate','min'),
    LastPurchase=('InvoiceDate','max')
).reset_index()

# RFM features
customer_agg['RecencyDays'] = (pd.Timestamp.now() - customer_agg['LastPurchase']).dt.days
customer_agg['TenureDays'] = (customer_agg['LastPurchase'] - customer_agg['FirstPurchase']).dt.days
customer_agg['TenureMonths'] = customer_agg['TenureDays'] / 30
customer_agg['FrequencyPerMonth'] = customer_agg['NumTransactions'] / customer_agg['TenureMonths']
customer_agg['MonetaryPerMonth'] = customer_agg['TotalSpent'] / customer_agg['TenureMonths']
customer_agg['AvgDaysBetweenInvoices'] = customer_agg['TenureDays'] / customer_agg['NumTransactions']

data = data.merge(customer_agg, on='CustomerID', how='left')

# product-level aggregation ---
product_agg = data.groupby('StockCode').agg(
    TotalQuantitySold=('Quantity','sum'),
    TotalRevenueGenerated=('TotalPrice','sum'),
    AvgUnitPriceSold=('UnitPrice','mean'),
    NumUniqueCustomers=('CustomerID','nunique')
).reset_index()

data = data.merge(product_agg, on='StockCode', how='left')

# interaction features

# price deviation
data['PricePerUnitDeviation'] = data['UnitPrice'] / data['AvgUnitPriceSold']

# customer-product affinity
cust_prod = data.groupby(['CustomerID','StockCode']).size().reset_index(name='CustProdCount')
data = data.merge(cust_prod, on=['CustomerID','StockCode'], how='left')
data['CustomerProductAffinity'] = data['CustProdCount'] / data['NumTransactions']

# invoice diversity
data['InvoiceDiversity'] = data['NumUniqueItems'] / data['TotalQuantity']

# country-level spending deviation
country_avg = data.groupby('Country')['TotalPrice'].mean().reset_index(name='CountryAvgSpend')
data = data.merge(country_avg, on='Country', how='left')
data['CountryAvgSpendDeviation'] = data['TotalPrice'] - data['CountryAvgSpend']


# recency-frequency-monetary interactions
data['Recency_x_Frequency'] = data['RecencyDays'] * data['FrequencyPerMonth']
data['Frequency_x_Monetary'] = data['FrequencyPerMonth'] * data['MonetaryPerMonth']

# delete temporary columns
data.drop(columns=['CustProdCount'], inplace=True)

# fill missing numeric values (just in case)
numeric_cols = data.select_dtypes(include=['float64','int64']).columns
data[numeric_cols] = data[numeric_cols].fillna(0)




## Semantic product clustering

In [12]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score

# generate embeddings
model = SentenceTransformer('all-MiniLM-L12-v2')
descriptions = data['Description']
embeddings = model.encode(descriptions, show_progress_bar=True)
embeddings = np.array(embeddings)


# normalize embeddings

scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

# PCA

pca = PCA(n_components=50, random_state=42)
embeddings_reduced = pca.fit_transform(embeddings_scaled)

# evaluate multiple cluster numbers

cluster_range = range(5, 31, 5)  # 5, 10, 15, 20, 25, 30 clusters
ch_scores = []
db_scores = []
sil_scores = []

for n_clusters in cluster_range:
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1024)
    labels = kmeans.fit_predict(embeddings_reduced)
    
    ch = calinski_harabasz_score(embeddings_reduced, labels)
    db = davies_bouldin_score(embeddings_reduced, labels)
    sil = silhouette_score(embeddings_reduced, labels)
    
    ch_scores.append(ch)
    db_scores.append(db)
    sil_scores.append(sil)
    
    print(f"n_clusters={n_clusters} -> CH={ch:.2f}, DB={db:.2f}, SIL={sil:.2f}")


# plot evaluation metrics

plt.figure(figsize=(12,5))
plt.plot(cluster_range, ch_scores, marker='o', label='Calinski-Harabasz (higher better)')
plt.plot(cluster_range, db_scores, marker='s', label='Davies-Bouldin (lower better)')
plt.plot(cluster_range, sil_scores, marker='^', label='Silhouette Score (higher better)')
plt.xlabel('Number of Clusters')
plt.ylabel('Score')
plt.title('KMeans Cluster Evaluation')
plt.legend()
plt.show()


# select best cluster number

best_index = np.argmax(np.array(ch_scores) / np.array(db_scores))
best_n_clusters = cluster_range[best_index]
print(f"\nRecommended number of clusters: {best_n_clusters}")


# fit final KMeans and assign clusters
kmeans_final = MiniBatchKMeans(n_clusters=best_n_clusters, random_state=42, batch_size=1024)
final_labels = kmeans_final.fit_predict(embeddings_reduced)

data['ProductClusterSemantic'] = final_labels



NameError: name 'descriptions' is not defined

In [ ]:
# plot cluster distribution
plt.figure(figsize=(12,6))
sns.countplot(x='ProductClusterSemantic', data=data, palette='tab20')
plt.title('Number of Products per Semantic Cluster (KMeans)')
plt.xlabel('Cluster')
plt.ylabel('Count')
plt.show()


# example descriptions per cluster
for cluster_id in sorted(data['ProductClusterSemantic'].unique()):
    print(f"\n--- Cluster {int(cluster_id)} ---")
    examples = data[data['ProductClusterSemantic'] == cluster_id]['Description'].head(5).tolist()
    for desc in examples:
        print(desc)


## Sentiment analysis to understand customer preferences

In [ ]:
from textblob import TextBlob
from tqdm.auto import tqdm

tqdm.pandas()


def get_sentiment(text):
    blob = TextBlob(str(text))
    polarity = blob.sentiment.polarity  # -1 (negative) to +1 (positive)
    
    if polarity > 0.1:
        sentiment = "POSITIVE"
    elif polarity < -0.1:
        sentiment = "NEGATIVE"
    else:
        sentiment = "NEUTRAL"
        
    return pd.Series([sentiment, polarity])


data[['Sentiment', 'SentimentScore']] = data['Description'].dropna().progress_apply(get_sentiment)

# Fill missing descriptions
data['Sentiment'].fillna('NEUTRAL', inplace=True)
data['SentimentScore'].fillna(0.0, inplace=True)


print(data[['Description', 'Sentiment', 'SentimentScore']].head())


In [ ]:
# detect deluxe/premium products 
data['IsDeluxe'] = data['Description'].str.contains(
    r'\b(deluxe|premium|exclusive|luxury|special edition)\b', 
    case=False, 
    na=False
)

mean_scores = data.groupby('IsDeluxe')['SentimentScore'].mean()
print(mean_scores)


In [ ]:
luxury_keywords = [
    'deluxe', 'premium', 'exclusive', 'luxury', 'special edition',
    'gold', 'silver', 'platinum', 'limited', 'gift', 'collection',
    'designer', 'high quality', 'elite', 'royal', 'signature'
]


pattern = r'\b(' + '|'.join(luxury_keywords) + r')\b'
data['IsDeluxe'] = data['Description'].str.contains(pattern, case=False, na=False)

deluxe_data = (
    data[data['IsDeluxe'] == True]
    .drop_duplicates(subset=['Description'])
    .sort_values(by='SentimentScore', ascending=False)
)

top_n = min(20, len(deluxe_data))
deluxe_data = deluxe_data.head(top_n)

fig = px.bar(
    deluxe_data,
    x='SentimentScore',
    y='Description',
    color='Description', 
    orientation='h',
    title=f'🛍️ Top {top_n} Deluxe/Premium Products by Sentiment',
    text='SentimentScore',
    color_discrete_sequence=px.colors.qualitative.Safe  
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_title='Sentiment Score',
    yaxis_title='Product Description',
    height=800,
    template='plotly_white',
    showlegend=False  
)

fig.show()


In [ ]:
# new features


# count of products bought from each semantic cluster

cluster_counts = (
    data.groupby(['CustomerID', 'ProductClusterSemantic'])
    .size()
    .unstack(fill_value=0)
    .add_prefix('ClusterCount_')
)

data = data.merge(cluster_counts, on='CustomerID', how='left')


# percentage of deluxe/premium products purchased

deluxe_ratio = (
    data.groupby('CustomerID')['IsDeluxe']
    .transform('mean')
)
data['DeluxeRatio'] = deluxe_ratio


# average sentiment of purchased products

avg_sentiment = (
    data.groupby('CustomerID')['SentimentScore']
    .transform('mean')
)
data['AvgSentimentScore'] = avg_sentiment


print(data[['CustomerID', 'DeluxeRatio', 'AvgSentimentScore'] + [c for c in data.columns if 'ClusterCount_' in c]].head())


## Encoding

In [43]:
# Get all unique dtypes in data
unique_dtypes = data.dtypes.unique()

# Convert to a list of type names
unique_dtype_names = [dt.name for dt in unique_dtypes]

print("Unique data types in data:")
print(unique_dtype_names)


Unique data types in data:
['object', 'int64', 'datetime64[ns]', 'float64', 'int32', 'bool']


In [44]:
data_orig = data.copy()
data_orig.to_csv('data_cleaned_orig.csv', sep=',', encoding='utf-8', index=False)




In [46]:
# Select columns that are object or categorical type
categorical_columns = [col for col in data.columns 
                       if data[col].dtype == 'object' 
                       or str(data[col].dtype) == 'category']

unique_counts = {}
for col in categorical_columns:
    unique_counts[col] = {
        "train_unique": data[col].nunique(),
    }

unique_counts


{'InvoiceNo': {'train_unique': 22190},
 'StockCode': {'train_unique': 3684},
 'Description': {'train_unique': 3896},
 'Country': {'train_unique': 37},
 'Sentiment': {'train_unique': 3}}

In [47]:
# ---- Label encode Sentiment ----
le = LabelEncoder()
data['SentimentEncoded'] = le.fit_transform(data['Sentiment'])

# ---- One-hot encode Country ----
data = pd.get_dummies(data, columns=['Country'], drop_first=True)

In [ ]:
# mean encoding for column StockCode to know which products are generally high-value
mean_target = data.groupby('StockCode')['TotalSpent'].mean()
data['StockCode_enc'] = data['StockCode'].map(mean_target)


In [ ]:
# each missing CustomerID gets a unique integer 
missing_idx = data[data['CustomerID'] == 0].index
data.loc[missing_idx, 'CustomerID'] += np.arange(len(missing_idx))


In [48]:
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])

# Extract features
data['InvoiceYear'] = data['InvoiceDate'].dt.year
data['InvoiceMonth'] = data['InvoiceDate'].dt.month
data['InvoiceDay'] = data['InvoiceDate'].dt.day
data['InvoiceWeekday'] = data['InvoiceDate'].dt.weekday  # 0=Monday
data['InvoiceHour'] = data['InvoiceDate'].dt.hour


In [55]:
for col in ['FirstPurchase', 'LastPurchase']:
    data[f'{col}_Year'] = data[col].dt.year
    data[f'{col}_Month'] = data[col].dt.month
    data[f'{col}_Day'] = data[col].dt.day
    data[f'{col}_Weekday'] = data[col].dt.weekday  # 0=Monday
    data[f'{col}_Hour'] = data[col].dt.hour


data['DaysSinceFirstPurchase'] = (data['LastPurchase'] - data['FirstPurchase']).dt.days


data = data.drop(columns=['FirstPurchase', 'LastPurchase'])


In [49]:
data = data.drop(columns=['InvoiceDate','InvoiceNo','StockCode','Description','Sentiment'])


In [50]:
data

,Quantity,UnitPrice,CustomerID,TotalPrice,DescriptionLength,NumWordsInDescription,IsReturn,InvoiceHour,InvoiceDayOfWeek,InvoiceMonth,...,Country_Spain,Country_Sweden,Country_Switzerland,Country_USA,Country_United Arab Emirates,Country_United Kingdom,Country_Unspecified,InvoiceYear,InvoiceDay,InvoiceWeekday
0,6,2.55,17850.0,15.30,34,5,0,8,2,12,...,False,False,False,False,False,True,False,2010,1,2
1,6,3.39,17850.0,20.34,19,3,0,8,2,12,...,False,False,False,False,False,True,False,2010,1,2
2,8,2.75,17850.0,22.00,30,5,0,8,2,12,...,False,False,False,False,False,True,False,2010,1,2
3,6,3.39,17850.0,20.34,35,6,0,8,2,12,...,False,False,False,False,False,True,False,2010,1,2
4,6,3.39,17850.0,20.34,30,5,0,8,2,12,...,False,False,False,False,False,True,False,2010,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401599,12,0.85,12680.0,10.20,27,5,0,12,4,12,...,False,False,False,False,False,False,False,2011,9,4
401600,6,2.10,12680.0,12.60,28,4,0,12,4,12,...,False,False,False,False,False,False,False,2011,9,4
401601,4,4.15,12680.0,16.60,29,4,0,12,4,12,...,False,False,False,False,False,False,False,2011,9,4
401602,4,4.15,12680.0,16.60,31,4,0,12,4,12,...,False,False,False,False,False,False,False,2011,9,4


## Scaling

In [58]:
num_bool_cols = data.select_dtypes(include=['int64', 'float64', 'int32', 'bool']).columns.tolist()

scaler = MinMaxScaler()


data[num_bool_cols] = scaler.fit_transform(data[num_bool_cols])




In [59]:
data

,Quantity,UnitPrice,CustomerID,TotalPrice,DescriptionLength,NumWordsInDescription,IsReturn,InvoiceHour,InvoiceDayOfWeek,InvoiceMonth,...,FirstPurchase_Month,FirstPurchase_Day,FirstPurchase_Weekday,FirstPurchase_Hour,LastPurchase_Year,LastPurchase_Month,LastPurchase_Day,LastPurchase_Weekday,LastPurchase_Hour,DaysSinceFirstPurchase
0,0.475,0.340000,0.926443,0.376000,0.965517,0.571429,0.0,0.142857,0.333333,1.0,...,1.000000,0.000000,0.333333,0.076923,1.0,0.090909,0.300000,0.500000,0.571429,0.190349
1,0.475,0.452000,0.926443,0.392800,0.448276,0.285714,0.0,0.142857,0.333333,1.0,...,1.000000,0.000000,0.333333,0.076923,1.0,0.090909,0.300000,0.500000,0.571429,0.190349
2,0.525,0.366667,0.926443,0.398333,0.827586,0.571429,0.0,0.142857,0.333333,1.0,...,1.000000,0.000000,0.333333,0.076923,1.0,0.090909,0.300000,0.500000,0.571429,0.190349
3,0.475,0.452000,0.926443,0.392800,1.000000,0.714286,0.0,0.142857,0.333333,1.0,...,1.000000,0.000000,0.333333,0.076923,1.0,0.090909,0.300000,0.500000,0.571429,0.190349
4,0.475,0.452000,0.926443,0.392800,0.827586,0.571429,0.0,0.142857,0.333333,1.0,...,1.000000,0.000000,0.333333,0.076923,1.0,0.090909,0.300000,0.500000,0.571429,0.190349
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401599,0.625,0.113333,0.056219,0.359000,0.724138,0.571429,0.0,0.428571,0.666667,1.0,...,0.636364,0.566667,0.500000,0.615385,1.0,1.000000,0.266667,0.666667,0.428571,0.300268
401600,0.475,0.280000,0.056219,0.367000,0.758621,0.428571,0.0,0.428571,0.666667,1.0,...,0.636364,0.566667,0.500000,0.615385,1.0,1.000000,0.266667,0.666667,0.428571,0.300268
401601,0.425,0.553333,0.056219,0.380333,0.793103,0.428571,0.0,0.428571,0.666667,1.0,...,0.636364,0.566667,0.500000,0.615385,1.0,1.000000,0.266667,0.666667,0.428571,0.300268
401602,0.425,0.553333,0.056219,0.380333,0.862069,0.428571,0.0,0.428571,0.666667,1.0,...,0.636364,0.566667,0.500000,0.615385,1.0,1.000000,0.266667,0.666667,0.428571,0.300268


In [60]:
data_scaled = data.copy()
data_scaled.to_csv('data_cleaned+scaled.csv', sep=',', encoding='utf-8', index=False)


